# Project Title: Intelligent Keyword Extraction System

In [1]:
# Cell 1: Install all required packages
!pip install -q keybert torch transformers gradio pyngrok nltk scikit-learn rake-nltk yake pandas plotly sentence-transformers pypdf2 python-docx wordcloud

print("✅ Installation complete!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.4/91.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 14.1 MB/s eta 0:00:00
✅ Installation complete!


In [2]:
# Cell 2: Configure ngrok
from pyngrok import ngrok

# Your ngrok authtoken
NGROK_AUTH_TOKEN = "371uOD6OqXzqZ8JNUUV8kMuIzK4_46t7rqxQejzni4QvfPKCZ"

# Configure ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("✅ ngrok configured successfully!")

✅ ngrok configured successfully!


In [3]:
# Cell 3: Import all necessary libraries
import gradio as gr
import pandas as pd
import numpy as np
from typing import List, Tuple, Dict
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Import keyword extraction models
from keybert import KeyBERT
from rake_nltk import Rake
import yake
from sklearn.feature_extraction.text import TfidfVectorizer
import torch

# Document parsing
import PyPDF2
from docx import Document
import os
from pathlib import Path

print("✅ All libraries imported successfully!")

# Load pre-trained models
print("🔄 Loading pre-trained models...")

# Load KeyBERT with lightweight model
try:
    kw_model = KeyBERT(model='paraphrase-MiniLM-L3-v2')
    print("✅ KeyBERT loaded (lightweight model)")
except:
    kw_model = KeyBERT(model='all-MiniLM-L6-v2')
    print("✅ KeyBERT loaded")

# Load RAKE
rake = Rake()
print("✅ RAKE loaded")

# Load YAKE
yake_extractor = yake.KeywordExtractor(lan="en", top=20, dedupLim=0.9)
print("✅ YAKE loaded")

print("✅ All models loaded successfully!")

✅ All libraries imported successfully!
🔄 Loading pre-trained models...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ KeyBERT loaded (lightweight model)
✅ RAKE loaded
✅ YAKE loaded
✅ All models loaded successfully!


In [4]:
# Cell 4: Core keyword extraction functions
class KeywordExtractor:
    def __init__(self):
        try:
            self.stop_words = set(stopwords.words('english'))
        except:
            self.stop_words = set(['the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by'])

    def preprocess_text(self, text: str) -> str:
        """Clean and preprocess input text"""
        text = re.sub(r'[^\w\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text)
        text = text.lower().strip()
        return text

    def extract_keybert(self, text: str, top_n: int = 10) -> List[Tuple[str, float]]:
        """Extract keywords using KeyBERT"""
        try:
            keywords = kw_model.extract_keywords(
                text,
                keyphrase_ngram_range=(1, 2),
                stop_words='english',
                top_n=top_n
            )
            return keywords
        except Exception as e:
            print(f"KeyBERT error: {e}")
            return []

    def extract_rake(self, text: str, top_n: int = 10) -> List[Tuple[str, float]]:
        """Extract keywords using RAKE"""
        try:
            rake.extract_keywords_from_text(text)
            ranked_phrases = rake.get_ranked_phrases_with_scores()
            return [(phrase, score) for score, phrase in ranked_phrases[:top_n]]
        except Exception as e:
            print(f"RAKE error: {e}")
            return []

    def extract_yake(self, text: str, top_n: int = 10) -> List[Tuple[str, float]]:
        """Extract keywords using YAKE"""
        try:
            keywords = yake_extractor.extract_keywords(text)
            return [(kw, score) for kw, score in keywords[:top_n]]
        except Exception as e:
            print(f"YAKE error: {e}")
            return []

    def extract_tfidf(self, text: str, top_n: int = 10) -> List[Tuple[str, float]]:
        """Extract keywords using TF-IDF"""
        try:
            sentences = sent_tokenize(text)
            if len(sentences) < 2:
                sentences = [text]

            vectorizer = TfidfVectorizer(
                ngram_range=(1, 2),
                stop_words='english',
                max_features=100
            )

            tfidf_matrix = vectorizer.fit_transform(sentences)
            feature_names = vectorizer.get_feature_names_out()

            avg_scores = tfidf_matrix.mean(axis=0).tolist()[0]
            keyword_scores = list(zip(feature_names, avg_scores))
            keyword_scores.sort(key=lambda x: x[1], reverse=True)

            return keyword_scores[:top_n]
        except Exception as e:
            print(f"TF-IDF error: {e}")
            return []

    def extract_hybrid(self, text: str, top_n: int = 10) -> List[Tuple[str, float]]:
        """Hybrid extraction combining multiple methods"""
        all_keywords = {}

        # KeyBERT (40% weight)
        keybert_kws = dict(self.extract_keybert(text, top_n=top_n))
        for kw, score in keybert_kws.items():
            all_keywords[kw] = all_keywords.get(kw, 0) + score * 0.4

        # RAKE (30% weight)
        rake_kws = dict(self.extract_rake(text, top_n=top_n))
        for kw, score in rake_kws.items():
            all_keywords[kw] = all_keywords.get(kw, 0) + (1 - min(score, 1.0)) * 0.3

        # YAKE (30% weight)
        yake_kws = dict(self.extract_yake(text, top_n=top_n))
        for kw, score in yake_kws.items():
            all_keywords[kw] = all_keywords.get(kw, 0) + (1 - min(score, 1.0)) * 0.3

        # Sort and return top N
        sorted_kws = sorted(all_keywords.items(), key=lambda x: x[1], reverse=True)
        return sorted_kws[:top_n]

# Initialize extractor
extractor = KeywordExtractor()
print("✅ Keyword extractor initialized!")

✅ Keyword extractor initialized!


In [5]:
# Cell 5: Document processing and visualization
def parse_uploaded_file(file):
    """Extract text from uploaded PDF, DOCX, or TXT file"""
    if file is None:
        return None, "No file uploaded"

    file_extension = Path(file.name).suffix.lower()
    text = ""

    try:
        if file_extension == '.txt':
            with open(file.name, 'r', encoding='utf-8') as f:
                text = f.read()
            return text, f"✅ Loaded: {Path(file.name).name}"

        elif file_extension == '.pdf':
            with open(file.name, 'rb') as f:
                pdf_reader = PyPDF2.PdfReader(f)
                for page in pdf_reader.pages:
                    text += page.extract_text()
            return text, f"✅ Loaded PDF: {Path(file.name).name}"

        elif file_extension == '.docx':
            doc = Document(file.name)
            for paragraph in doc.paragraphs:
                text += paragraph.text + "\n"
            return text, f"✅ Loaded DOCX: {Path(file.name).name}"

        else:
            return None, f"❌ Unsupported format: {file_extension}"

    except Exception as e:
        return None, f"❌ Error: {str(e)}"

def process_document(text: str, method: str, num_keywords: int) -> Dict:
    """Process document and extract keywords"""
    cleaned_text = extractor.preprocess_text(text)

    if method == "KeyBERT (BERT-based)":
        keywords = extractor.extract_keybert(cleaned_text, num_keywords)
    elif method == "RAKE":
        keywords = extractor.extract_rake(cleaned_text, num_keywords)
    elif method == "YAKE":
        keywords = extractor.extract_yake(cleaned_text, num_keywords)
    elif method == "TF-IDF":
        keywords = extractor.extract_tfidf(cleaned_text, num_keywords)
    else:
        keywords = extractor.extract_hybrid(cleaned_text, num_keywords)

    return {
        'keywords': keywords if keywords else [],
        'text_length': len(text),
        'word_count': len(text.split()),
        'sentence_count': len(sent_tokenize(text)),
        'method': method
    }

def create_visualization(keywords: List[Tuple[str, float]]):
    """Create interactive bar chart"""
    import plotly.express as px
    import plotly.graph_objects as go

    if not keywords:
        fig = go.Figure()
        fig.add_annotation(text="No keywords to display", x=0.5, y=0.5, showarrow=False)
        fig.update_layout(height=400)
        return fig

    kw_names = [kw[:30] for kw, _ in keywords]
    kw_scores = [score for _, score in keywords]

    fig = px.bar(
        x=kw_scores,
        y=kw_names,
        orientation='h',
        title='Top Keywords by Relevance Score',
        labels={'x': 'Relevance Score', 'y': 'Keywords'},
        color=kw_scores,
        color_continuous_scale='Viridis'
    )

    fig.update_layout(height=500, showlegend=False, title_x=0.5)
    return fig

print("✅ Processing functions ready!")

✅ Processing functions ready!


In [6]:
# Cell 6: Create professional Gradio interface
def keyword_extraction_interface(text=None, file=None, method="Hybrid (All methods)", num_keywords=10):
    """Main interface function"""

    # Handle file upload
    if file is not None:
        extracted_text, message = parse_uploaded_file(file)
        if extracted_text is None:
            return message, "⚠️ Processing failed", None, message
        text = extracted_text
        file_status = message
    else:
        file_status = "📝 Text input"

    if not text or text.strip() == "":
        return "❌ Please enter text or upload a document.", file_status, None, "No content"

    try:
        results = process_document(text, method, num_keywords)

        # Format keywords
        if results['keywords']:
            keywords_text = "### 🎯 Extracted Keywords\n\n"
            for i, (kw, score) in enumerate(results['keywords'], 1):
                emoji = "🔥" if score > 0.7 else "⭐" if score > 0.4 else "📌"
                keywords_text += f"{i}. {emoji} **{kw}** (score: {score:.4f})\n"
        else:
            keywords_text = "⚠️ No keywords extracted. Try a different method."

        # Statistics
        stats_text = f"""### 📊 Document Statistics
- **Source:** {file_status}
- **Method:** {results['method']}
- **Characters:** {results['text_length']:,}
- **Words:** {results['word_count']:,}
- **Sentences:** {results['sentence_count']}
- **Keywords:** {len(results['keywords'])}"""

        if results['keywords']:
            avg_score = sum(s for _, s in results['keywords']) / len(results['keywords'])
            stats_text += f"\n- **Avg Score:** {avg_score:.4f}"

        # Visualization
        fig = create_visualization(results['keywords'])

        # Summary
        if results['keywords']:
            top_keywords = [kw for kw, _ in results['keywords'][:5]]
            summary = f"**💡 Top Keywords:** {', '.join(top_keywords)}"
        else:
            summary = "**⚠️ No keywords extracted**"

        return keywords_text, stats_text, fig, summary

    except Exception as e:
        error_msg = f"❌ Error: {str(e)}"
        return error_msg, file_status, None, error_msg

# Create interface
with gr.Blocks(theme=gr.themes.Soft(), title="NLP Keyword Extraction") as demo:
    gr.Markdown("""
    # 🔍 Professional NLP Keyword Extraction System
    ### Extract keywords from documents using state-of-the-art NLP models
    """)

    with gr.Row():
        with gr.Column(scale=2):
            gr.Markdown("### 📁 Upload Document")
            file_input = gr.File(label="PDF, DOCX, or TXT", file_types=[".pdf", ".docx", ".txt"], type="filepath")

            gr.Markdown("### ✏️ Or Enter Text")
            text_input = gr.Textbox(label="Document Text", placeholder="Paste text here...", lines=8)

            with gr.Row():
                method_dropdown = gr.Dropdown(
                    choices=["KeyBERT (BERT-based)", "RAKE", "YAKE", "TF-IDF", "Hybrid (All methods)"],
                    value="Hybrid (All methods)",
                    label="Extraction Method"
                )

                num_keywords = gr.Slider(minimum=5, maximum=30, value=10, step=1, label="Number of Keywords")

            submit_btn = gr.Button("🚀 Extract Keywords", variant="primary", size="lg")

            with gr.Accordion("📝 Examples", open=False):
                gr.Examples(
                    examples=[
                        ["Artificial Intelligence (AI) and Machine Learning (ML) are transforming technology. Deep learning, neural networks, and natural language processing enable computers to understand and generate human-like text. These technologies power chatbots, recommendation systems, and autonomous vehicles."],
                        ["Climate change refers to long-term shifts in temperatures and weather patterns. Global warming, greenhouse gas emissions, and carbon dioxide are major concerns. Renewable energy sources like solar, wind, and hydroelectric power offer sustainable solutions."],
                    ],
                    inputs=text_input
                )

        with gr.Column(scale=1):
            output_keywords = gr.Markdown(label="🎯 Keywords")
            output_stats = gr.Markdown(label="📊 Statistics")

    with gr.Row():
        output_plot = gr.Plot(label="📈 Visualization")
        output_summary = gr.Markdown(label="💡 Summary")

    submit_btn.click(
        fn=keyword_extraction_interface,
        inputs=[text_input, file_input, method_dropdown, num_keywords],
        outputs=[output_keywords, output_stats, output_plot, output_summary]
    )

    gr.Markdown("""
    ---
    **Features:** KeyBERT, RAKE, YAKE, TF-IDF, Hybrid | **File Support:** PDF, DOCX, TXT | **Interactive Visualizations**
    """)

print("✅ Interface created successfully!")

✅ Interface created successfully!


In [7]:
# Cell 7: Launch application
import socket

def get_free_port():
    sock = socket.socket()
    sock.bind(('', 0))
    port = sock.getsockname()[1]
    sock.close()
    return port

# Kill existing tunnels
ngrok.kill()

# Create tunnel
port = get_free_port()
public_url = ngrok.connect(port).public_url
print(f"✅ ngrok tunnel established!")
print(f"🌍 Public URL: {public_url}")
print(f"🔗 Local URL: http://localhost:{port}")

# Launch
print("\n🚀 Launching application...")
demo.launch(server_port=port, share=False)

print("\n🎉 Application is running!")
print("📱 Access via the URL above")
print("⚠️ Press Ctrl+C to stop")

✅ ngrok tunnel established!
🌍 Public URL: https://idioblastic-fisher-wordy.ngrok-free.dev
🔗 Local URL: http://localhost:43131

🚀 Launching application...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>


🎉 Application is running!
📱 Access via the URL above
⚠️ Press Ctrl+C to stop


# README.md: Intelligent Keyword Extraction System

## 💡 Project Overview

The **Intelligent Keyword Extraction System** is a powerful tool designed to automatically identify and extract key terms and phrases from various types of documents (PDFs, DOCX, TXT, and plain text). Leveraging state-of-the-art Natural Language Processing (NLP) models, this system helps users quickly grasp the core content of lengthy texts, making information retrieval and content analysis significantly more efficient.

## ✨ How It Works

Our system employs a hybrid approach, combining multiple advanced keyword extraction techniques:

*   **KeyBERT (BERT-based):** Utilizes transformer models to find keywords that are semantically similar to the document's content.
*   **RAKE (Rapid Automatic Keyword Extraction):** A statistical algorithm that identifies keywords by analyzing word frequency and co-occurrence.
*   **YAKE (Yet Another Keyword Extractor):** Focuses on linguistic features of text, such as word casing, frequency, and position.
*   **TF-IDF (Term Frequency-Inverse Document Frequency):** A classic method that highlights words important to a document within a collection.

By blending these methods, the system provides a robust and comprehensive set of keywords, ranked by relevance. Users can upload documents or paste text directly into a user-friendly Gradio interface, receiving instant insights and interactive visualizations of the extracted keywords.

## 🎯 Where Is It Useful?

This system can be invaluable across various domains:

*   **Content Analysis:** Quickly summarize articles, reports, and research papers.
*   **SEO Optimization:** Identify relevant keywords for website content and marketing strategies.
*   **Academic Research:** Pinpoint critical concepts in academic literature.
*   **Legal Document Review:** Expedite the review of contracts and legal briefs by highlighting key clauses.
*   **Business Intelligence:** Extract insights from customer feedback, market research, and competitive analysis.
*   **Data Tagging:** Automatically tag documents for better organization and searchability.
*   **News Aggregation:** Summarize news articles and identify trending topics.

## 💰 Where Can It Be Sold?

The Intelligent Keyword Extraction System has strong market potential as a standalone product or an integrated service:

*   **SaaS Offering:** Deploy as a web application with subscription tiers for different usage levels.
*   **API Service:** Offer keyword extraction as an API for developers to integrate into their own applications (e.g., content management systems, research platforms).
*   **Enterprise Solutions:** Customize and license to large organizations for internal document processing and knowledge management.
*   **Gradio Hosting Platforms:** Deploy directly on platforms like Hugging Face Spaces or other cloud providers that support Gradio apps.
*   **Freelance/Consulting:** Offer specialized keyword extraction services for clients in content creation, marketing, or research.

## 🚀 Future Work

*   **Multi-language Support:** Extend keyword extraction capabilities to languages other than English.
*   **Advanced Semantic Understanding:** Incorporate more sophisticated NLP techniques for deeper contextual understanding and entity recognition.
*   **Custom Model Training:** Allow users to train models on their domain-specific datasets for even more accurate keyword extraction.
*   **Integration with Cloud Storage:** Seamless integration with Google Drive, Dropbox, or other cloud storage services for document input.
*   **Real-time Processing:** Optimize for near real-time keyword extraction from live data streams.
*   **User Feedback Loop:** Implement mechanisms for users to provide feedback on keyword quality, helping to refine the models over time.
*   **Automated Summarization:** Expand beyond keywords to generate concise summaries of documents.
*   **Enhanced Visualization:** Develop more interactive and insightful visualizations, such as keyword clouds or relationship graphs.

# README.md: Intelligent Keyword Extraction System

## 💡 Project Overview

The **Intelligent Keyword Extraction System** is a powerful tool designed to automatically identify and extract key terms and phrases from various types of documents (PDFs, DOCX, TXT, and plain text). Leveraging state-of-the-art Natural Language Processing (NLP) models, this system helps users quickly grasp the core content of lengthy texts, making information retrieval and content analysis significantly more efficient.

## ✨ How It Works

Our system employs a hybrid approach, combining multiple advanced keyword extraction techniques:

*   **KeyBERT (BERT-based):** Utilizes transformer models to find keywords that are semantically similar to the document's content.
*   **RAKE (Rapid Automatic Keyword Extraction):** A statistical algorithm that identifies keywords by analyzing word frequency and co-occurrence.
*   **YAKE (Yet Another Keyword Extractor):** Focuses on linguistic features of text, such as word casing, frequency, and position.
*   **TF-IDF (Term Frequency-Inverse Document Frequency):** A classic method that highlights words important to a document within a collection.

By blending these methods, the system provides a robust and comprehensive set of keywords, ranked by relevance. Users can upload documents or paste text directly into a user-friendly Gradio interface, receiving instant insights and interactive visualizations of the extracted keywords.

## 🎯 Where Is It Useful?

This system can be invaluable across various domains:

*   **Content Analysis:** Quickly summarize articles, reports, and research papers.
*   **SEO Optimization:** Identify relevant keywords for website content and marketing strategies.
*   **Academic Research:** Pinpoint critical concepts in academic literature.
*   **Legal Document Review:** Expedite the review of contracts and legal briefs by highlighting key clauses.
*   **Business Intelligence:** Extract insights from customer feedback, market research, and competitive analysis.
*   **Data Tagging:** Automatically tag documents for better organization and searchability.
*   **News Aggregation:** Summarize news articles and identify trending topics.

## 💰 Where Can It Be Sold?

The Intelligent Keyword Extraction System has strong market potential as a standalone product or an integrated service:

*   **SaaS Offering:** Deploy as a web application with subscription tiers for different usage levels.
*   **API Service:** Offer keyword extraction as an API for developers to integrate into their own applications (e.g., content management systems, research platforms).
*   **Enterprise Solutions:** Customize and license to large organizations for internal document processing and knowledge management.
*   **Gradio Hosting Platforms:** Deploy directly on platforms like Hugging Face Spaces or other cloud providers that support Gradio apps.
*   **Freelance/Consulting:** Offer specialized keyword extraction services for clients in content creation, marketing, or research.

## 🚀 Future Work

*   **Multi-language Support:** Extend keyword extraction capabilities to languages other than English.
*   **Advanced Semantic Understanding:** Incorporate more sophisticated NLP techniques for deeper contextual understanding and entity recognition.
*   **Custom Model Training:** Allow users to train models on their domain-specific datasets for even more accurate keyword extraction.
*   **Integration with Cloud Storage:** Seamless integration with Google Drive, Dropbox, or other cloud storage services for document input.
*   **Real-time Processing:** Optimize for near real-time keyword extraction from live data streams.
*   **User Feedback Loop:** Implement mechanisms for users to provide feedback on keyword quality, helping to refine the models over time.
*   **Automated Summarization:** Expand beyond keywords to generate concise summaries of documents.
*   **Enhanced Visualization:** Develop more interactive and insightful visualizations, such as keyword clouds or relationship graphs.